# ch05 Bonus 03：超参搜索（Hyperparameter Tuning）

> 对照官方 `ch05/05_bonus_hparam-tuning`

## 一句话

学习率、batch size、权重衰减等超参对训练效果影响巨大。本 notebook 用**网格搜索**找一组让 loss 最低的超参组合。

## 方法

在固定小训练预算下，遍历几组候选超参（如 lr ∈ {1e-4, 3e-4, 1e-3}），每组跑几轮训练，挑 loss 最低的。真实场景常用更高效的贝叶斯优化（Optuna）。

In [ ]:
import torch
import torch.nn.functional as F
from pathlib import Path
from src.gpt import GPTModel, GPT_CONFIG_124M, create_dataloader_v1

# 准备数据（小配置）
text = Path("data/the-verdict.txt").read_text(encoding="utf-8")
cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 128, "n_layers": 2, "n_heads": 4, "context_length": 128})
dl = create_dataloader_v1(text, batch_size=2, max_length=cfg["context_length"],
                          stride=cfg["context_length"], shuffle=True, drop_last=True)

def quick_train(lr, wd, epochs=2):
    """用给定超参快速训练 2 轮，返回最终 loss。"""
    torch.manual_seed(123)
    model = GPTModel(cfg)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    model.train()
    final = 0.0
    for _ in range(epochs):
        total = 0; n = 0
        for x, y in dl:
            opt.zero_grad()
            loss = F.cross_entropy(model(x).flatten(0,1), y.flatten())
            loss.backward(); opt.step()
            total += loss.item(); n += 1
        final = total / n
    return final

In [ ]:
# 网格搜索：3 个学习率 × 2 个权重衰减
lrs = [1e-4, 3e-4, 1e-3]
wds = [0.0, 0.1]
results = {}

print(f"{'lr':<8} {'wd':<6} {'loss':<10}")
print("-" * 26)
for lr in lrs:
    for wd in wds:
        loss = quick_train(lr, wd)
        results[(lr, wd)] = loss
        print(f"{lr:<8} {wd:<6} {loss:<10.4f}")

best = min(results, key=results.get)
print(f"\n✓ 最佳: lr={best[0]}, wd={best[1]}, loss={results[best]:.4f}")
print("💡 真实场景用 Optuna 做贝叶斯优化，搜索空间更大更高效。")